In [1]:
import numpy as np
import re
import pandas as pd

In [2]:
df=pd.read_csv('gurgaon_properties_cleaned_v1.csv')

In [3]:
# pd.set_option('display.max_columns',None)
# pd.set_option('display.max_rows',None)
# pd.set_option('display.max_colwidth',None)

In [4]:
df.head(1)

,property_type,society,sector,price_in_crore,price_per_sqft,area,areaWithType,bedRoom,bathroom,balcony,additionalRoom,floorNum,facing,agePossession,nearbyLocations,furnishDetails,features
0,flat,smriti apartment,56,0.45,7500.0,600.0,Super Built up area 600(55.74 sq.m.),1,1,2,not available,3.0,NaN,10+ Year Old,"['Sector 54 chowk metro station', 'Sector metr...",[],"['Power Back-up', 'Feng Shui / Vaastu Complian..."


## 1. areaWithType

In [5]:
df[['area','areaWithType']].sample(5)

,area,areaWithType
3342,2000.0,Plot area 2200(204.39 sq.m.)Built Up area: 200...
3614,2542.0,Carpet area: 2542 (236.16 sq.m.)
2288,2495.0,Super Built up area 2905(269.88 sq.m.)Carpet a...
3328,1350.0,Super Built up area 2191(203.55 sq.m.)Built Up...
2663,1425.0,Super Built up area 1425(132.39 sq.m.)Built Up...


In [6]:
# this function extracts super built up area
def get_super_built_up_area(text):
    match=re.search(r'Super Built up area (\d+\.?\d*)',text)
    if match:
        return float(match.group(1))
    return None

In [7]:
#this function extracts the built up area or carpet area 
def get_area(text, area_type):
    match=re.search(area_type+r'\s*:\s*(\d+\.?\d+)',text)
    if match:
        return float(match.group(1))
    return None

In [8]:
#this function chcks if the area is provided in sq.m. and converts it to sqrt if needed
def convert_to_sqft(text,area_value):
    if area_value is None:
        return None
    match=re.search(r'{}\((\d+\.?\d*) sq.m.\)'.format(area_value),text)
    if match:
        sq_m_value=float(match.group(1))
        return sq_m_value*10.7639
    return area_value

In [9]:
#extract super built up area and convert to sqft if needed
df['super_built_up_area']=df['areaWithType'].apply(get_super_built_up_area)
df['super_built_up_area']=df.apply(lambda x:convert_to_sqft(x['areaWithType'],x['super_built_up_area']),axis=1)
#extract built up area and convert to sqft if needed
df['built_up_area']=df['areaWithType'].apply(lambda x:get_area(x,'Built Up area'))
df['built_up_area']=df.apply(lambda x:convert_to_sqft(x['areaWithType'],x['built_up_area']),axis=1)
#extract carpet area and convert to sqft if needed
df['carpet_area']=df['areaWithType'].apply(lambda x:get_area(x,'Carpet area'))
df['carpet_area']=df.apply(lambda x:convert_to_sqft(x['areaWithType'],x['carpet_area']),axis=1)


In [10]:
df.head()

,property_type,society,sector,price_in_crore,price_per_sqft,area,areaWithType,bedRoom,bathroom,balcony,additionalRoom,floorNum,facing,agePossession,nearbyLocations,furnishDetails,features,super_built_up_area,built_up_area,carpet_area
0,flat,smriti apartment,56,0.45,7500.0,600.0,Super Built up area 600(55.74 sq.m.),1,1,2,not available,3.0,NaN,10+ Year Old,"['Sector 54 chowk metro station', 'Sector metr...",[],"['Power Back-up', 'Feng Shui / Vaastu Complian...",600.0,NaN,NaN
1,flat,signature global park,48,0.54,7200.0,750.0,Carpet area: 750 (69.68 sq.m.),2,2,2,not available,4.0,North-East,Under Construction,"['Sector 55-56 metro', 'Global city centre', '...",NaN,"['Security / Fire Alarm', 'Feng Shui / Vaastu ...",NaN,NaN,750.0
2,flat,ats triumph,104,1.85,8078.0,2290.0,Super Built up area 2290(212.75 sq.m.),3,4,3+,"servant room,study room,pooja room,store room",14.0,East,1 to 5 Year Old,"['IFFCO Chowk Metro Station', 'The Esplanade M...","['7 Fan', '7 Light', '1 Modular Kitchen', 'No ...","['Security / Fire Alarm', 'Feng Shui / Vaastu ...",2290.0,NaN,NaN
3,flat,pareena laxmi apartments,99,0.32,6106.0,524.0,Super Built up area 524(48.68 sq.m.)Carpet are...,2,2,2,not available,2.0,NaN,0 to 1 Year Old,"['Dwarka Expy', 'Govt. Sr. Sec. School', 'Exce...",[],"['Feng Shui / Vaastu Compliant', 'Lift(s)', 'S...",524.0,NaN,424.8
4,flat,m3m woodshire,107,1.40,5929.0,2361.0,Super Built up area 2361(219.34 sq.m.),3,4,0,not available,1.0,East,0 to 1 Year Old,NaN,"['1 Light', 'No AC', 'No Bed', 'No Chimney', '...","['Power Back-up', 'Intercom Facility', 'Lift(s...",2361.0,NaN,NaN
